In [4]:


import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report,confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
from sympy.physics.quantum.matrixutils import matrix_zeros
from win32con import PROFILE_KERNEL

In [5]:
# handling Path for the notebook

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "players_features.csv"

df = pd.read_csv(DATA_PATH)

df.shape

(2029, 8)

In [6]:
df = df.replace([np.inf, -np.inf], np.nan)
print("NaNs per column : ")
print(df.isna().sum())

NaNs per column : 
age_years                 0
min                       0
availability_ratio        0
offensive_efficiency    128
progressive_actions     438
position                  0
compition                 0
value_tier                0
dtype: int64


In [7]:
zero_fill_cols = [
    'offensive_efficiency',
    'progressive_actions'
]
df[zero_fill_cols] = df[zero_fill_cols].fillna(0)

print(df.isna().sum())

age_years               0
min                     0
availability_ratio      0
offensive_efficiency    0
progressive_actions     0
position                0
compition               0
value_tier              0
dtype: int64


In [8]:
# Splitting Features and Target

X = df.drop(columns=["value_tier"])
y = df['value_tier']

numeric_features = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

numeric_features,categorical_features

(['age_years',
  'min',
  'availability_ratio',
  'offensive_efficiency',
  'progressive_actions'],
 ['position', 'compition'])

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y , test_size = 0.2, random_state = 42, stratify = y)

X_train.shape, X_test.shape

((1623, 7), (406, 7))

In [10]:
# Preprocessing Pipeline

from sklearn.impute import SimpleImputer
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [11]:
baseline_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)

print("---Logistic Regression---")
print(classification_report(y_test, y_pred_baseline))

---Logistic Regression---
              precision    recall  f1-score   support

        High       0.96      0.99      0.97       136
         Low       0.89      0.87      0.88       135
         Mid       0.86      0.85      0.86       135

    accuracy                           0.90       406
   macro avg       0.90      0.90      0.90       406
weighted avg       0.90      0.90      0.90       406



C:\Users\arvin\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
rf_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(n_estimators=300,random_state=42,class_weight='balanced'))

    ]
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("---Random Forest---")
print(classification_report(y_test, y_pred_rf))

---Random Forest---
              precision    recall  f1-score   support

        High       0.95      0.96      0.95       136
         Low       0.90      0.90      0.90       135
         Mid       0.85      0.85      0.85       135

    accuracy                           0.90       406
   macro avg       0.90      0.90      0.90       406
weighted avg       0.90      0.90      0.90       406



In [15]:
from sklearn.metrics import accuracy_score

def train_and_evaluate(X,y,drop_features=None):
    if drop_features:
        X = X.drop(columns=drop_features)

    num_feats = X.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_feats = X.select_dtypes(include=['object']).columns.tolist()

    numeric_transformer = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="median"))]
    )

    categorical_transformer = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="most_frequent")),
               ('onehot', OneHotEncoder(handle_unknown="ignore"))]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_feats),
            ("cat", categorical_transformer, cat_feats),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", LogisticRegression(max_iter=1000))
        ]
    )

    X_train, X_test, y_train, y_test = train_test_split(X,y , test_size = 0.2, random_state = 42,stratify = y)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return accuracy_score(y_test,preds)


In [16]:
baseline_acc = train_and_evaluate(X,y)
baseline_acc

C:\Users\arvin\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.9039408866995073

In [17]:
# removing availability and checking accuracy
acc_no_availability = train_and_evaluate(
    X,y, drop_features=['min','availability_ratio']
)
acc_no_availability

C:\Users\arvin\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.6847290640394089

In [18]:
# removing age and calculating accuracy
acc_no_age = train_and_evaluate(X,y, drop_features=['age_years'])
acc_no_age

C:\Users\arvin\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7635467980295566

In [19]:
#Removing performance matrics and calculating accuracy
acc_no_performance = train_and_evaluate(X,y, drop_features=['offensive_efficiency','progressive_actions'])

acc_no_performance

C:\Users\arvin\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.7487684729064039

In [20]:
#Removing position and league

acc_no_context = train_and_evaluate(X,y, drop_features=['position','compition'])
acc_no_context

C:\Users\arvin\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8891625615763546

In [21]:
# Summary of the results

ablation_results = pd.DataFrame({
    "Scenario": ['All  Features','No availablity',"No age","No performance",'No context'],
    "Accuracy":[baseline_acc,acc_no_availability,acc_no_age,acc_no_performance,acc_no_context]
})

ablation_results

,Scenario,Accuracy
0,All Features,0.903941
1,No availablity,0.684729
2,No age,0.763547
3,No performance,0.748768
4,No context,0.889163
